# 03 — SCD2: история изменений клиентов (банковский канон)
SCD2 как в банковских DWH:- **`hash_diff`** — изменения по хэшу атрибутов (канон Data Vault 2.0)- **`valid_to = 5999-12-31`** — дата-заглушка вместо NULL- **смежные интервалы** `[valid_from, valid_to)`: дата закрытия старой = дата открытия новой- **MERGE по пути** `delta.\`s3a://...\`` — без metastoreЗапускать после `02_silver` (нужен `silver/clients`).

## 1. Spark-сессия и константы

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("scd2").getOrCreate()

SCD_PATH = "s3a://silver/clients_scd2"
MAX_DATE = "5999-12-31"
ATTRS = ["city", "segment"]

def with_hash_diff(df):
    return df.withColumn("hash_diff", F.md5(F.concat_ws("||", *[F.col(a) for a in ATTRS])))

print("Spark:", spark.version)

Spark: 3.5.3


## 2. Первая загрузка: инициализируем SCD2
`valid_from` = дата регистрации, `valid_to` = `5999-12-31`, `is_current` = true,`hash_diff` = отпечаток атрибутов.

In [2]:
clients = spark.read.format("delta").load("s3a://silver/clients")

scd_init = (with_hash_diff(
        clients.select("client_id", "full_name", "city", "segment",
                       F.col("reg_date").alias("valid_from")))
    .withColumn("valid_to", F.lit(MAX_DATE).cast("date"))
    .withColumn("is_current", F.lit(True)))

(scd_init.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(SCD_PATH))

cnt = spark.read.format("delta").load(SCD_PATH).count()
print("SCD2 инициализирована:", cnt, "версий")

SCD2 инициализирована: 9828 версий


## 3. Вторая загрузка: пришли изменения

**Почему изменения «искусственные»?** В реальном банке источник делает выгрузку
регулярно (раз в день), и каждая новая выгрузка естественно содержит обновления —
клиент переехал, сменил сегмент. Наш генератор делает **один** снимок, поэтому
второй выгрузки с изменениями физически нет. Чтобы показать механику SCD2,
мы **имитируем** её здесь: берём текущих клиентов и у части (`client_id <= 500`)
меняем город на «Сочи» и сегмент на «private». В проде эти изменения пришли бы
из источника сами — мы лишь воспроизводим сценарий.

Выбор детерминированный (по `client_id`, без `rand()`), чтобы результат был
воспроизводимым. Выгрузку материализуем во временный parquet, отвязывая её
от Delta-таблицы, которую будем менять.

In [3]:
# берём текущие актуальные версии
current = (spark.read.format("delta").load(SCD_PATH)
    .filter("is_current = true")
    .select("client_id", "full_name", "city", "segment"))

# детерминированно: первые 500 client_id «переехали»
updates = (current
    .withColumn("city",
        F.when(F.col("client_id") <= 500, F.lit("Сочи")).otherwise(F.col("city")))
    .withColumn("segment",
        F.when(F.col("client_id") <= 500, F.lit("private")).otherwise(F.col("segment"))))
updates = with_hash_diff(updates)

# отвязываем от Delta-таблицы: пишем во временный parquet и читаем обратно
updates.write.mode("overwrite").parquet("s3a://silver/_tmp_updates")
updates = spark.read.parquet("s3a://silver/_tmp_updates")
updates.createOrReplaceTempView("updates")

changed_cnt = updates.filter("city = 'Сочи'").count()
print("В новой выгрузке:", updates.count(), "клиентов, изменено:", changed_cnt)

В новой выгрузке: 9828 клиентов, изменено: 488


## 4. MERGE шаг 1: закрываем изменившиеся версии
Сравнение по `hash_diff`. Закрываем старую версию: `is_current=false`,`valid_to=сегодня` (= дата открытия новой, смежные интервалы).

In [4]:
spark.sql(f'''
    MERGE INTO delta.`{SCD_PATH}` AS t
    USING updates AS s
    ON t.client_id = s.client_id AND t.is_current = true
    WHEN MATCHED AND t.hash_diff <> s.hash_diff THEN
        UPDATE SET t.is_current = false,
                   t.valid_to = current_date()
''')

closed = spark.read.format("delta").load(SCD_PATH).filter("is_current = false").count()
print("Закрыто старых версий:", closed)

Закрыто старых версий: 488


## 5. MERGE шаг 2: вставляем новые версии
Новые версии нужны тем, у кого хэш изменился. Определяем их **прямым сравнениемвыгрузки с закрытыми версиями** (а не left_anti по живой таблице) — берём извыгрузки тех, для кого в таблице есть только что закрытая версия с таким hash.

In [5]:
# client_id, у которых только что закрыли версию сегодняшней датой
closed_today = (spark.read.format("delta").load(SCD_PATH)
    .filter("is_current = false AND valid_to = current_date()")
    .select("client_id").distinct())

# им нужна новая актуальная версия — берём данные из выгрузки
new_versions = (updates.join(closed_today, "client_id", "inner")
    .withColumn("valid_from", F.current_date())
    .withColumn("valid_to", F.lit(MAX_DATE).cast("date"))
    .withColumn("is_current", F.lit(True))
    .select("client_id", "full_name", "city", "segment",
            "valid_from", "valid_to", "is_current", "hash_diff"))

(new_versions.write.format("delta").mode("append").save(SCD_PATH))
print("Вставлено новых версий:", new_versions.count())

Вставлено новых версий: 488


## 6. Проверка: история изменившегося клиента

In [6]:
scd_all = spark.read.format("delta").load(SCD_PATH)

print("Всего версий:", scd_all.count())
print("Актуальных:", scd_all.filter("is_current = true").count())
print()

# клиент 1 точно «переехал» (client_id <= 500)
print("=== История клиента 1: было -> стало ===")
(scd_all.filter(F.col("client_id") == 1)
    .select("client_id", "city", "segment", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from")
    .show(truncate=False))

Всего версий: 10316
Актуальных: 9828

=== История клиента 1: было -> стало ===
+---------+------+--------+----------+----------+----------+
|client_id|city  |segment |valid_from|valid_to  |is_current|
+---------+------+--------+----------+----------+----------+
|1        |Казань|affluent|2021-09-29|2026-05-24|false     |
|1        |Сочи  |private |2026-05-24|5999-12-31|true      |
+---------+------+--------+----------+----------+----------+



## Готово
Каноничный SCD2: `hash_diff`, открытая версия `valid_to=5999-12-31`,смежные интервалы (дата закрытия старой = дата открытия новой).Дальше — `04_gold`: бизнес-витрины.